# Task 5 — Filter Testing on HLK-LD2413 Water Level Data

**Pipeline:**
1. **Hard Rejection** — replace known-bad readings with `NaN` based on error codes
2. **Resample** — reindex to a uniform 15-min grid
3. **Filter** — each filter fills NaN gaps and smooths the signal
4. **Evaluate** — compare against `filtered_data.csv` (manual ground truth) using RMSE / MAE


In [ ]:
%matplotlib tk

import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import warnings
warnings.filterwarnings('ignore')

# ── PATHS ──────────────────────────────────────────────────────────────────────
BASE      = r"c:\Users\sahil\Desktop\ICFOSS\Anomaly Detection"
CSV_RAW   = BASE + r"\data\combined_data.csv"
CSV_TRUTH = BASE + r"\data\filtered_data.csv"

# ── PARAMETERS ─────────────────────────────────────────────────────────────────
WINDOW        = 5      # rolling window (5 x 15-min = 75 min)
EMA_SPAN      = 5
HAMPEL_HALF   = 7      # half-window for Hampel identifier
HAMPEL_SIGMA  = 3.0    # k * MAD threshold
MAX_ROC       = 0.5    # max delta-WL per 15-min step (metres)

# ── LOAD & HARD REJECT ─────────────────────────────────────────────────────────
df = pd.read_csv(CSV_RAW)
df['Time']        = pd.to_datetime(df['Time'], format='%d-%m-%Y %H:%M')
df['Water Level'] = pd.to_numeric(df['Water Level'], errors='coerce')
df['errorcode']   = pd.to_numeric(df['errorcode'],   errors='coerce').astype(int)
df = df.sort_values('Time').reset_index(drop=True)

mask_ec1      = df['errorcode'] == 1
mask_ec3      = df['errorcode'] == 3
mask_ec5_zero = (df['errorcode'] == 5) & (df['Water Level'] == 0)

df['rejected'] = mask_ec1 | mask_ec3 | mask_ec5_zero
df['WL_raw']   = df['Water Level'].copy()
df.loc[df['rejected'], 'Water Level'] = np.nan

# ── RESAMPLE TO UNIFORM 15-MIN GRID (ASOF MERGE) ───────────────────────────────
# Create uniform 15-minute grid
start_grid = df['Time'].min().round('15min')
end_grid = df['Time'].max().round('15min')
grid_index = pd.date_range(start=start_grid, end=end_grid, freq='15min')
grid_df = pd.DataFrame({'GridTime': grid_index})

# Align using merge_asof with 7-min tolerance to handle alignment shifts/drifts
df_sorted = df.sort_values('Time')
df = pd.merge_asof(
    grid_df,
    df_sorted,
    left_on='GridTime',
    right_on='Time',
    direction='nearest',
    tolerance=pd.Timedelta(minutes=7)
)

# Preserve original unrounded Time for plotting, fill gaps with GridTime
df['Time_plot'] = df['Time'].fillna(df['GridTime'])
df['OriginalTime'] = df['Time']
df['Time'] = df['Time_plot']

# ── GROUND TRUTH ───────────────────────────────────────────────────────────────
gt = pd.read_csv(CSV_TRUTH)
gt['Time']        = pd.to_datetime(gt['Time'], format='%d-%m-%Y %H:%M')
gt['Water Level'] = pd.to_numeric(gt['Water Level'], errors='coerce')
gt = gt.sort_values('Time').reset_index(drop=True)
print(f"15-min grid slots : {len(grid_df)}")
print(f"NaN slots         : {int(df['Water Level'].isna().sum())}")
print(f"Ground truth rows : {len(gt)}")


## Helper Functions

In [3]:
def metrics(pred_series, df_raw, gt):
    """RMSE, MAE, MaxErr vs ground truth aligned on common timestamps."""
    df_p   = pd.DataFrame({'Time': df_raw['Time'], 'pred': pred_series.values})
    merged = pd.merge(df_p,
                      gt[['Time', 'Water Level']].rename(columns={'Water Level': 'truth'}),
                      on='Time', how='inner').dropna()
    if merged.empty:
        return dict(RMSE=float('nan'), MAE=float('nan'), MaxErr=float('nan'), N=0)
    err = merged['pred'] - merged['truth']
    return dict(RMSE=float(np.sqrt((err**2).mean())),
                MAE=float(err.abs().mean()),
                MaxErr=float(err.abs().max()),
                N=len(merged))


def plot_filter(df_raw, gt, filtered_series, filter_name, color, m):
    """Single-panel plot: raw data vs ground truth vs filter output. No PNG saved."""
    fig, ax = plt.subplots(figsize=(15, 5))

    ax.plot(df_raw['Time'], df_raw['WL_raw'],
            color='#555555', alpha=0.40, lw=0.7,
            label='Raw (combined_data.csv)', zorder=1)
    ax.plot(gt['Time'], gt['Water Level'],
            color='#2ecc71', alpha=0.85, lw=1.4, ls='--',
            label='Ground truth (filtered_data.csv)', zorder=2)
    ax.plot(df_raw['Time'], filtered_series,
            color=color, alpha=0.92, lw=1.8,
            label=f'{filter_name} output', zorder=3)

    ax.set_title(
        f'Task 5 — {filter_name}\n'
        f'RMSE={m["RMSE"]:.4f} m  |  MAE={m["MAE"]:.4f} m  |  '
        f'Max Error={m["MaxErr"]:.4f} m  |  N={m["N"]} matched pts',
        fontsize=11, fontweight='bold'
    )
    ax.set_ylabel('Water Level (m)', fontsize=11)
    ax.set_xlabel('Date', fontsize=11)
    ax.set_ylim(-0.5, 5.2)
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%d-%b'))
    ax.xaxis.set_major_locator(mdates.AutoDateLocator())
    ax.legend(fontsize=9, loc='upper right', framealpha=0.92)
    fig.autofmt_xdate()
    plt.tight_layout()
    plt.show(block=False)
    print(f'{filter_name}: RMSE={m["RMSE"]:.4f}  MAE={m["MAE"]:.4f}  MaxErr={m["MaxErr"]:.4f}  N={m["N"]}')


## Filter 1 — Moving Average

**How it works:** Linearly interpolates NaN gaps, then applies a centred rolling mean (window = 5 steps = 75 min).

**Characteristic:** Smooth but symmetric lag — it blurs sharp edges of real tide changes.


In [3]:
wl_ma = (df['Water Level']
         .interpolate(method='linear', limit_direction='both')
         .rolling(window=WINDOW, center=True, min_periods=1).mean())

m_ma = metrics(wl_ma, df, gt)
plot_filter(df, gt, wl_ma, 'Moving Average', '#e74c3c', m_ma)


Moving Average: RMSE=0.0617  MAE=0.0277  MaxErr=0.9540  N=8000


## Filter 2 — Median Filter

**How it works:** Interpolates NaN gaps, then applies a centred rolling median (window = 5 steps).

**Characteristic:** More robust than MA to isolated spikes — the median ignores single extreme values in the window.


In [6]:
wl_med = (df['Water Level']
          .interpolate(method='linear', limit_direction='both')
          .rolling(window=WINDOW, center=True, min_periods=1).median())

m_med = metrics(wl_med, df, gt)
plot_filter(df, gt, wl_med, 'Median Filter', '#9b59b6', m_med)


Median Filter: RMSE=0.0304  MAE=0.0061  MaxErr=0.9200  N=8000


## Filter 3 — Exponential Moving Average (EMA)

**How it works:** Interpolates NaN gaps, then applies Exponential Weighted Mean (span = 5, causal).

**Characteristic:** Causal filter — faster response to rising values but has lag on the trailing edge.


In [27]:
wl_ema = (df['Water Level']
          .interpolate(method='linear', limit_direction='both')
          .ewm(span=EMA_SPAN, adjust=False).mean())

m_ema = metrics(wl_ema, df, gt)
plot_filter(df, gt, wl_ema, 'EMA (Exponential Moving Average)', '#f39c12', m_ema)


EMA (Exponential Moving Average): RMSE=0.1529  MAE=0.0852  MaxErr=1.0506  N=8000


## Filter 4 — Hampel Filter

**How it works:**
1. Interpolate NaN gaps
2. Slide a window; compute median and MAD
3. Points beyond σ × 1.4826 × MAD from local median → NaN
4. Interpolate again to fill detected outliers

**Characteristic:** Statistical outlier detection beyond hard-rejection rules.


In [7]:
k_mad = 1.4826
wl_h = df['Water Level'].interpolate(method='linear', limit_direction='both').copy()
n = len(wl_h)
outlier_mask = pd.Series(False, index=wl_h.index)

for i in range(n):
    lo = max(0, i - HAMPEL_HALF)
    hi = min(n, i + HAMPEL_HALF + 1)
    win = wl_h.iloc[lo:hi]
    med = win.median()
    mad = (win - med).abs().median()
    if abs(wl_h.iloc[i] - med) > HAMPEL_SIGMA * k_mad * mad:
        outlier_mask.iloc[i] = True

wl_h[outlier_mask] = np.nan
wl_hamp = wl_h.interpolate(method='linear', limit_direction='both')

print(f"Hampel detected {int(outlier_mask.sum())} additional statistical outliers")
m_hamp = metrics(wl_hamp, df, gt)
plot_filter(df, gt, wl_hamp, 'Hampel Filter', '#e84393', m_hamp)


Hampel detected 152 additional statistical outliers
Hampel Filter: RMSE=0.0189  MAE=0.0008  MaxErr=0.9350  N=8000


## Filter 5 — Rate-of-Change (RoC) Limiter

**How it works:**
1. Interpolate NaN gaps
2. Walk forward: if |WL[i] - WL[i-1]| > MAX_ROC, clamp WL[i] to WL[i-1] ± MAX_ROC

**Characteristic:** Physically-motivated — water cannot jump > 0.5 m in 15 minutes under normal conditions.


In [5]:
arr = df['Water Level'].interpolate(method='linear', limit_direction='both').values.copy()
clamp_count = 0
for i in range(1, len(arr)):
    if np.isnan(arr[i]) or np.isnan(arr[i-1]):
        continue
    delta = arr[i] - arr[i-1]
    if abs(delta) > MAX_ROC:
        arr[i] = arr[i-1] + np.sign(delta) * MAX_ROC
        clamp_count += 1

wl_roc = pd.Series(arr, index=df.index)
print(f"RoC Limiter clamped {clamp_count} steps exceeding {MAX_ROC} m/step")
m_roc = metrics(wl_roc, df, gt)
plot_filter(df, gt, wl_roc, 'Rate-of-Change Limiter', '#2980b9', m_roc)


RoC Limiter clamped 33 steps exceeding 0.5 m/step
Rate-of-Change Limiter: RMSE=0.0125  MAE=0.0005  MaxErr=0.5600  N=8000


## Filter 6 — 2D Kalman Filter

**How it works:**
1. Interpolates NaN gaps.
2. Tracks both Water Level (position) and its Rate of Change (velocity).

**Characteristic:** Effective at tracking true changes while resisting large sudden spikes by balancing process noise and measurement noise.


In [10]:
# 2D Kalman Filter
dt = 1.0
F = np.array([[1, dt], [0, 1]])
H = np.array([[1, 0]])

q = 0.05
Q = np.array([[q*(dt**3)/3, q*(dt**2)/2], 
              [q*(dt**2)/2, q*dt]])
R = np.array([[0.5]])

wl_k = df['Water Level'].interpolate(method='linear', limit_direction='both').copy()
x = np.array([[wl_k.iloc[0]], [0.0]])
P = np.eye(2) * 1.0

filtered_vals = []
for i in range(len(wl_k)):
    x_pred = F.dot(x)
    P_pred = F.dot(P).dot(F.T) + Q
    
    z = np.array([[wl_k.iloc[i]]])
    y = z - H.dot(x_pred)
    S = H.dot(P_pred).dot(H.T) + R
    K = P_pred.dot(H.T).dot(np.linalg.inv(S))
    
    x = x_pred + K.dot(y)
    P = (np.eye(2) - K.dot(H)).dot(P_pred)
    
    filtered_vals.append(x[0, 0])

wl_kalman = pd.Series(filtered_vals, index=df.index)

m_kalman = metrics(wl_kalman, df, gt)
plot_filter(df, gt, wl_kalman, '2D Kalman Filter', '#16a085', m_kalman)


2D Kalman Filter: RMSE=0.0703  MAE=0.0374  MaxErr=0.5957  N=8000


## Summary — All Filters Compared

In [ ]:
all_metrics = {
    'Moving Average': m_ma,
    'Median Filter':  m_med,
    'EMA':            m_ema,
    'Hampel Filter':  m_hamp,
    'RoC Limiter':    m_roc,
    '2D Kalman':      m_kalman,
}

print()
print("=" * 65)
print("  TASK 5 -- FILTER PERFORMANCE SUMMARY")
print("  (vs filtered_data.csv as ground truth)")
print("=" * 65)
print(f"  {'Filter':<24} {'RMSE':>8} {'MAE':>8} {'MaxErr':>9} {'N':>7}")
print(f"  {'-'*24} {'-'*8} {'-'*8} {'-'*9} {'-'*7}")
for name, m in all_metrics.items():
    print(f"  {name:<24} {m['RMSE']:>8.4f} {m['MAE']:>8.4f} {m['MaxErr']:>9.4f} {m['N']:>7}")
print("=" * 65)

best_rmse = min(all_metrics, key=lambda k: all_metrics[k]['RMSE'])
best_mae  = min(all_metrics, key=lambda k: all_metrics[k]['MAE'])
print(f"\n  >> Best by RMSE : {best_rmse}  ({all_metrics[best_rmse]['RMSE']:.4f} m)")
print(f"  >> Best by MAE  : {best_mae}  ({all_metrics[best_mae]['MAE']:.4f} m)")

# ── Overlay: all filters on one plot ──────────────────────────────────────────
fig, ax = plt.subplots(figsize=(15, 5))

ax.plot(df['Time'], df['WL_raw'],
        color='#555555', alpha=0.20, lw=0.5, label='Raw data', zorder=1)
ax.plot(gt['Time'], gt['Water Level'],
        color='#2ecc71', alpha=0.75, lw=1.5, ls='--',
        label='Ground truth', zorder=2)

filter_series = {
    'Moving Average': (wl_ma,   '#e74c3c'),
    'Median Filter':  (wl_med,  '#9b59b6'),
    'EMA':            (wl_ema,  '#f39c12'),
    'Hampel Filter':  (wl_hamp, '#e84393'),
    'RoC Limiter':    (wl_roc,  '#2980b9'),
    '2D Kalman':      (wl_kalman, '#16a085'),
}
for name, (series, color) in filter_series.items():
    ax.plot(df['Time'], series, color=color, alpha=0.80, lw=1.1,
            label=f'{name} (RMSE={all_metrics[name]["RMSE"]:.4f}m)', zorder=3)

ax.set_title('Task 5 — All Filters Compared', fontsize=13, fontweight='bold')
ax.set_ylabel('Water Level (m)', fontsize=11)
ax.set_xlabel('Date', fontsize=11)
ax.set_ylim(-0.5, 5.2)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%d-%b'))
ax.legend(fontsize=8, loc='upper right', framealpha=0.95, ncol=2)
fig.autofmt_xdate()
plt.tight_layout()
plt.show(block=False)
